# Automated Web Security Scanner
**Author:** Aditya Pathak  
**Purpose:** Passive checks of HTTPS use, common HTTP security headers, and basic CORS response headers.

> **Authorization:** Run this only against a website you own or have explicit permission to assess. This notebook makes a normal HTTP GET request and does not exploit or brute-force anything.

## How to use
1. Install dependencies with `pip install -r requirements.txt`.
2. Set `TARGET_URL` below to an authorized test website.
3. Run all cells from top to bottom.
4. Optionally save the results as JSON.

A missing header is a hardening opportunity, not proof that a site is vulnerable. CORS needs contextual validation.


In [ ]:
from urllib.parse import urlparse
from datetime import datetime, timezone
import json
import requests
from pprint import pprint

SECURITY_HEADERS = {
    "Strict-Transport-Security": "Helps enforce HTTPS in supported browsers.",
    "Content-Security-Policy": "Helps reduce risks from injected content.",
    "X-Content-Type-Options": "Prevents MIME-type sniffing when set to nosniff.",
    "X-Frame-Options": "Helps reduce clickjacking risk; CSP frame-ancestors is a modern alternative.",
    "Referrer-Policy": "Controls how much referrer information browsers send.",
    "Permissions-Policy": "Restricts access to selected browser features.",
}

def normalize_url(url):
    url = url.strip()
    if not url:
        raise ValueError("Please provide a target URL.")
    if "://" not in url:
        url = "https://" + url
    parsed = urlparse(url)
    if parsed.scheme not in ("http", "https") or not parsed.hostname:
        raise ValueError("Use an http:// or https:// URL with a hostname.")
    if parsed.username or parsed.password:
        raise ValueError("Do not include credentials in the URL.")
    return url

def scan_url(url, timeout=8):
    target = normalize_url(url)
    report = {
        "target": target,
        "checked_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "status_code": None,
        "final_url": None,
        "https_used": target.lower().startswith("https://"),
        "redirected": False,
        "headers": {},
        "cors": {},
        "errors": [],
    }
    try:
        response = requests.get(
            target, timeout=timeout, allow_redirects=True,
            headers={"User-Agent": "PassiveWebSecurityScanner/1.0"}
        )
        report["status_code"] = response.status_code
        report["final_url"] = response.url
        report["redirected"] = bool(response.history)
        report["https_used"] = response.url.lower().startswith("https://")
        for name, note in SECURITY_HEADERS.items():
            value = response.headers.get(name)
            report["headers"][name] = {
                "present": value is not None,
                "value": value or "",
                "note": note
            }
        origin = response.headers.get("Access-Control-Allow-Origin")
        credentials = response.headers.get("Access-Control-Allow-Credentials")
        report["cors"] = {
            "allow_origin_present": origin is not None,
            "allow_origin": origin or "",
            "allow_credentials": credentials or "",
            "wildcard_with_credentials": origin == "*" and str(credentials).lower() == "true",
            "note": "A CORS header alone does not establish an exploitable flaw."
        }
    except requests.RequestException as exc:
        report["errors"].append(f"{exc.__class__.__name__}: {exc}")
    return report

def display_report(report):
    print("PASSIVE WEB SECURITY SCAN")
    print("Target:", report["target"])
    print("UTC time:", report["checked_at_utc"])
    print("HTTP status:", report["status_code"])
    print("Final URL:", report["final_url"])
    print("HTTPS at final URL:", report["https_used"])
    print("Redirected:", report["redirected"])
    print("\nSecurity headers:")
    for name, item in report["headers"].items():
        print(f"  {name}: {'PRESENT' if item['present'] else 'MISSING'}")
        if item["present"]:
            print("    Value:", item["value"])
    print("\nCORS observations:")
    pprint(report["cors"])
    if report["errors"]:
        print("\nErrors:")
        for err in report["errors"]:
            print("-", err)
    print("\nInterpretation: missing headers are recommendations for review, not proof of a vulnerability.")

print("Scanner functions loaded.")

## Run a scan
Change the URL to a domain you own or have explicit permission to test. For a quick demo, you can use a public documentation site, but avoid scanning systems without authorization.

In [ ]:
TARGET_URL = "https://example.com"  # Replace with an authorized target
TIMEOUT_SECONDS = 8

report = scan_url(TARGET_URL, timeout=TIMEOUT_SECONDS)
display_report(report)

## Save results (optional)
The next cell writes the findings to `scan_report.json` in the notebook's current folder.

In [ ]:
with open("scan_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)
print("Saved scan_report.json")

## Limitations
- Performs a single ordinary GET request and follows redirects.
- Does not test authentication, JavaScript behavior, TLS cipher suites, or exploitability.
- Some applications set headers conditionally by route or response type.
- A human should validate findings before reporting a vulnerability.
